# 📘 Databricks SQL & Dashboards
## SQL Warehouses, Query Editor, Lakeview Dashboards & BI Integration

**What you'll learn:**
- Databricks SQL overview and use cases
- SQL Warehouse types and sizing
- Query editor features (autocomplete, history, parameters)
- Lakeview Dashboards (formerly SQL Dashboards)
- Alerts and scheduled queries
- BI tool integration (Tableau, Power BI, Looker)
- Query optimization for SQL Warehouses
- Cost management for analyst workloads

**Prerequisites:** Notebooks 01 (SQL), 33 (Platform)
**Study Order:** 33

---

---
# 📗 Section 1: Databricks SQL Overview

## What is Databricks SQL?

Databricks SQL (DBSQL) is the BI-optimized interface for running SQL queries
against your lakehouse. It's designed for analysts, not engineers.

| Feature | All-Purpose Cluster | SQL Warehouse |
|---------|-------------------|---------------|
| **Optimized for** | Development (Python, Scala) | SQL queries (BI) |
| **Start time** | 3-7 minutes | Seconds (serverless) |
| **Concurrency** | Limited | High (auto-scales) |
| **Query caching** | Manual | Automatic |
| **BI connectivity** | Possible but not optimized | Native (JDBC/ODBC) |
| **Cost** | Per-minute (even idle) | Per-query (serverless) |

## SQL Warehouse Sizing

| Size | vCPUs | Memory | Concurrent Queries | Best For |
|------|-------|--------|-------------------|----------|
| 2X-Small | 4 | 16 GB | 2-4 | Single analyst |
| X-Small | 8 | 32 GB | 4-8 | Small team |
| Small | 16 | 64 GB | 8-16 | Medium team |
| Medium | 32 | 128 GB | 16-32 | Large team |
| Large | 64 | 256 GB | 32-64 | Enterprise |

## Lakeview Dashboards

Lakeview (formerly SQL Dashboards) provides:
- Drag-and-drop dashboard builder
- Auto-refresh on schedule
- Parameterized queries
- Sharing with stakeholders
- Embedding in other tools

```sql
-- Parameterized query for dashboard
SELECT 
    date_trunc('day', order_date) AS day,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue
FROM gold.daily_orders
WHERE order_date >= :start_date
  AND order_date <= :end_date
  AND region = :region
GROUP BY 1
ORDER BY 1
```

In [0]:
# SQL patterns optimized for Databricks SQL Warehouses
print("📊 SQL WAREHOUSE OPTIMIZATION PATTERNS")
print("=" * 60)

patterns = {
    "Use Liquid Clustering": {
        "bad": "SELECT * FROM orders WHERE date = '2024-03-15' -- full scan!",
        "good": "CREATE TABLE orders CLUSTER BY (date, region); -- then same query is fast",
        "impact": "10-100x faster for filtered queries"
    },
    "Avoid SELECT *": {
        "bad": "SELECT * FROM orders WHERE customer_id = 42",
        "good": "SELECT order_id, total_amount, status FROM orders WHERE customer_id = 42",
        "impact": "Read only needed columns (columnar storage benefit)"
    },
    "Use Query Result Cache": {
        "bad": "Run same expensive query 100 times (100x cost)",
        "good": "First run computes, next 99 served from cache (1x cost)",
        "impact": "Automatic in SQL Warehouses — no code change needed"
    },
    "Parameterized Queries": {
        "bad": "Hardcode dates: WHERE date = '2024-03-15'",
        "good": "Use parameters: WHERE date = :report_date",
        "impact": "Reusable queries, dashboard filters, cache-friendly"
    },
}

for name, details in patterns.items():
    print(f"\n📌 {name}")
    print(f"   ❌ Bad:  {details['bad']}")
    print(f"   ✅ Good: {details['good']}")
    print(f"   Impact: {details['impact']}")

---
# 📗 Section 2: SQL Warehouses

## What is a SQL Warehouse?

A SQL Warehouse is compute optimized for SQL queries and BI workloads.
Unlike all-purpose clusters, SQL Warehouses:
- Start in seconds (serverless) or minutes (classic)
- Auto-scale for concurrent users
- Auto-suspend when idle
- Optimized for SQL (not Python/Scala)

## Warehouse Types

| Type | Start Time | Best For |
|------|-----------|---------|
| **Serverless** | Seconds | Most use cases (recommended) |
| **Pro** | 2-3 minutes | Advanced features, larger workloads |
| **Classic** | 3-5 minutes | Legacy, specific configurations |

## Sizing Guide

| Size | vCPUs | Concurrent Queries | Best For |
|------|-------|-------------------|---------|
| 2X-Small | 4 | 2-4 | Single analyst |
| Small | 8 | 4-8 | Small team (3-5) |
| Medium | 16 | 8-16 | Medium team (10-20) |
| Large | 32 | 16-32 | Large team (20-50) |

## Lakeview Dashboards

Lakeview (formerly SQL Dashboards) provides:
- Drag-and-drop dashboard builder
- Auto-refresh on schedule
- Parameterized queries (filter by date, region, etc.)
- Sharing with stakeholders (view-only links)
- Embedding in other tools

```sql
-- Parameterized query for dashboard
SELECT
    DATE_TRUNC('day', order_date) AS day,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue
FROM gold.daily_orders
WHERE order_date >= :start_date
  AND order_date <= :end_date
  AND region = :region
GROUP BY 1
ORDER BY 1
```

In [0]:
%sql
-- SQL Warehouse optimization patterns

-- 1. Use Liquid Clustering for fast filtered queries
-- CREATE TABLE gold.daily_revenue CLUSTER BY (order_date, region)

-- 2. Pre-aggregate for dashboard queries
SELECT
    order_date,
    customer_segment,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(total_amount), 2) AS revenue,
    ROUND(AVG(total_amount), 2) AS avg_order_value
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY order_date, customer_segment
ORDER BY order_date DESC, revenue DESC
LIMIT 100

In [0]:
%sql
-- 3. Query result caching (automatic in SQL Warehouses)
-- Same query run twice: second run is instant (from cache)

-- 4. Parameterized queries (use :param_name syntax)
-- In Lakeview dashboards, these become filter widgets:
SELECT
    product_name,
    category,
    SUM(quantity) AS units_sold,
    SUM(line_total) AS revenue
FROM techmart_dw.order_items oi
JOIN techmart_dw.products p ON oi.product_id = p.product_id
JOIN techmart_dw.orders o ON oi.order_id = o.order_id
WHERE o.order_date >= '2024-01-01'
  AND p.category = 'Electronics'
GROUP BY product_name, category
ORDER BY revenue DESC
LIMIT 20

## Best Practices for SQL Dashboards

| Practice | Why |
|----------|-----|
| Pre-aggregate in Gold layer | Avoid full scans on every dashboard load |
| Use Liquid Clustering | 10-100x faster filtered queries |
| Set auto-suspend (10 min) | Avoid paying for idle warehouse |
| Use query result cache | Repeated queries are instant |
| Limit result sets | Don't return millions of rows to browser |
| Schedule refreshes off-peak | Reduce cost, avoid peak contention |

---
*Notebook 33 of the Databricks Data Engineering series*
*Study Order: 33*

---
# 📗 Section 2: SQL Warehouses

## What is a SQL Warehouse?

A SQL Warehouse is compute optimized for SQL queries and BI workloads.
Unlike all-purpose clusters, SQL Warehouses:
- Start in seconds (serverless) or minutes (classic)
- Auto-scale for concurrent users
- Auto-suspend when idle
- Optimized for SQL (not Python/Scala)

## Warehouse Types

| Type | Start Time | Best For |
|------|-----------|---------|
| **Serverless** | Seconds | Most use cases (recommended) |
| **Pro** | 2-3 minutes | Advanced features, larger workloads |
| **Classic** | 3-5 minutes | Legacy, specific configurations |

## Sizing Guide

| Size | vCPUs | Concurrent Queries | Best For |
|------|-------|-------------------|---------|
| 2X-Small | 4 | 2-4 | Single analyst |
| Small | 8 | 4-8 | Small team (3-5) |
| Medium | 16 | 8-16 | Medium team (10-20) |
| Large | 32 | 16-32 | Large team (20-50) |

## Lakeview Dashboards

Lakeview (formerly SQL Dashboards) provides:
- Drag-and-drop dashboard builder
- Auto-refresh on schedule
- Parameterized queries (filter by date, region, etc.)
- Sharing with stakeholders (view-only links)
- Embedding in other tools

```sql
-- Parameterized query for dashboard
SELECT
    DATE_TRUNC('day', order_date) AS day,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue
FROM gold.daily_orders
WHERE order_date >= :start_date
  AND order_date <= :end_date
  AND region = :region
GROUP BY 1
ORDER BY 1
```

In [0]:
%sql
-- SQL Warehouse optimization patterns

-- 1. Use Liquid Clustering for fast filtered queries
-- CREATE TABLE gold.daily_revenue CLUSTER BY (order_date, region)

-- 2. Pre-aggregate for dashboard queries
SELECT
    order_date,
    customer_segment,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(total_amount), 2) AS revenue,
    ROUND(AVG(total_amount), 2) AS avg_order_value
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY order_date, customer_segment
ORDER BY order_date DESC, revenue DESC
LIMIT 100

In [0]:
%sql
-- 3. Query result caching (automatic in SQL Warehouses)
-- Same query run twice: second run is instant (from cache)

-- 4. Parameterized queries (use :param_name syntax)
-- In Lakeview dashboards, these become filter widgets:
SELECT
    product_name,
    category,
    SUM(quantity) AS units_sold,
    SUM(line_total) AS revenue
FROM techmart_dw.order_items oi
JOIN techmart_dw.products p ON oi.product_id = p.product_id
JOIN techmart_dw.orders o ON oi.order_id = o.order_id
WHERE o.order_date >= '2024-01-01'
  AND p.category = 'Electronics'
GROUP BY product_name, category
ORDER BY revenue DESC
LIMIT 20

## Best Practices for SQL Dashboards

| Practice | Why |
|----------|-----|
| Pre-aggregate in Gold layer | Avoid full scans on every dashboard load |
| Use Liquid Clustering | 10-100x faster filtered queries |
| Set auto-suspend (10 min) | Avoid paying for idle warehouse |
| Use query result cache | Repeated queries are instant |
| Limit result sets | Don't return millions of rows to browser |
| Schedule refreshes off-peak | Reduce cost, avoid peak contention |

---
*Notebook 33 of the Databricks Data Engineering series*
*Study Order: 33*

---
# 📗 Section 3: Query Optimization for Dashboards

## Pre-Aggregation Pattern

The most important optimization: pre-compute expensive aggregations in Gold layer.

```sql
-- Instead of this (runs on every dashboard load):
SELECT
    DATE_TRUNC('day', o.order_date) AS day,
    c.customer_segment,
    p.category,
    COUNT(*) AS orders,
    SUM(oi.line_total) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY 1, 2, 3

-- Pre-compute this in Gold (runs once per day):
CREATE TABLE gold.daily_segment_category_revenue AS
SELECT ... (same query above)

-- Dashboard query (instant!):
SELECT * FROM gold.daily_segment_category_revenue
WHERE day >= :start_date
```

## Lakeview Dashboard Features

- **Filters**: Date pickers, dropdowns, text search
- **Visualizations**: Bar, line, pie, scatter, table, counter
- **Auto-refresh**: Set refresh interval (5 min, 1 hour, daily)
- **Sharing**: Share with view-only link or embed in other tools
- **Alerts**: Notify when a metric crosses a threshold

In [0]:
%sql
-- Build a dashboard-ready Gold table
DROP TABLE IF EXISTS techmart_dw.gold_dashboard_metrics;

CREATE TABLE techmart_dw.gold_dashboard_metrics AS
SELECT
    o.order_date,
    c.customer_segment,
    p.category AS product_category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    ROUND(SUM(oi.line_total), 2) AS total_revenue,
    ROUND(AVG(oi.line_total), 2) AS avg_line_value
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
JOIN techmart_dw.order_items oi ON o.order_id = oi.order_id
JOIN techmart_dw.products p ON oi.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY o.order_date, c.customer_segment, p.category

In [0]:
%sql
-- Dashboard query (fast -- reads pre-aggregated Gold table)
SELECT
    customer_segment,
    product_category,
    SUM(total_orders) AS orders,
    SUM(total_revenue) AS revenue,
    ROUND(SUM(total_revenue) / SUM(total_orders), 2) AS avg_order_value
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= '2024-01-01'
GROUP BY customer_segment, product_category
ORDER BY revenue DESC
LIMIT 20

---
# Section 4: Lakeview Dashboards

## Building Dashboards in Databricks

Lakeview (formerly SQL Dashboards) provides a drag-and-drop dashboard builder.

### Dashboard Components

| Component | Description | Example |
|-----------|-------------|---------|
| **Counter** | Single metric | Total Revenue: $1.2M |
| **Bar chart** | Compare categories | Revenue by region |
| **Line chart** | Trends over time | Daily orders (30 days) |
| **Pie chart** | Proportions | Revenue by payment method |
| **Table** | Detailed data | Top 20 customers |
| **Filter** | Interactive controls | Date range, region picker |

### Parameterized Queries

```sql
-- Parameters become filter widgets in the dashboard
SELECT
    order_date,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue
FROM techmart_dw.orders
WHERE order_date BETWEEN :start_date AND :end_date
  AND status IN (:status_filter)
  AND customer_segment = :segment
GROUP BY order_date
ORDER BY order_date
```

### Auto-Refresh

```
Dashboard settings -> Schedule -> Every 1 hour
```

The dashboard automatically re-runs all queries on schedule.
Analysts always see fresh data without manual refresh.

### Sharing Dashboards

- **View-only link**: Share with stakeholders (no Databricks account needed)
- **Embed**: Embed in Confluence, Notion, or internal portals
- **Schedule email**: Send PDF snapshot daily/weekly
- **Alerts**: Notify when a metric crosses a threshold

In [0]:
%sql
-- Build a comprehensive Gold table for dashboards
-- This pre-aggregates data so dashboards load instantly

DROP TABLE IF EXISTS techmart_dw.gold_dashboard_metrics;

CREATE TABLE techmart_dw.gold_dashboard_metrics AS
SELECT
    o.order_date,
    YEAR(o.order_date) AS order_year,
    MONTH(o.order_date) AS order_month,
    c.customer_segment,
    p.category AS product_category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    ROUND(SUM(oi.line_total), 2) AS total_revenue,
    ROUND(AVG(oi.line_total), 2) AS avg_line_value,
    ROUND(SUM(oi.line_total) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
JOIN techmart_dw.order_items oi ON o.order_id = oi.order_id
JOIN techmart_dw.products p ON oi.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY o.order_date, YEAR(o.order_date), MONTH(o.order_date),
         c.customer_segment, p.category

In [0]:
%sql
-- Dashboard query 1: Revenue trend (last 30 days)
SELECT order_date, SUM(total_revenue) AS revenue, SUM(total_orders) AS orders
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= DATE_ADD(CURRENT_DATE(), -30)
GROUP BY order_date
ORDER BY order_date

In [0]:
%sql
-- Dashboard query 2: Revenue by segment and category
SELECT
    customer_segment,
    product_category,
    SUM(total_revenue) AS revenue,
    SUM(total_orders) AS orders
FROM techmart_dw.gold_dashboard_metrics
GROUP BY customer_segment, product_category
ORDER BY revenue DESC
LIMIT 20

In [0]:
%sql
-- Dashboard query 3: KPI summary (counter widgets)
SELECT
    SUM(total_revenue) AS total_revenue,
    SUM(total_orders) AS total_orders,
    COUNT(DISTINCT unique_customers) AS unique_customers,
    ROUND(SUM(total_revenue) / SUM(total_orders), 2) AS avg_order_value
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= DATE_ADD(CURRENT_DATE(), -30)

---
# Section 5: SQL Warehouse Optimization for Dashboards

## Key Optimizations

1. **Pre-aggregate in Gold layer** -- avoid full scans on every load
2. **Liquid Clustering** on filter columns (date, region, segment)
3. **Query result cache** -- automatic in SQL Warehouses (same query = instant)
4. **Auto-suspend** -- set to 10 minutes to avoid idle cost
5. **Serverless warehouse** -- instant start, no cluster management

## Warehouse Sizing Guide

| Team Size | Warehouse Size | Concurrent Queries |
|-----------|---------------|-------------------|
| 1-3 analysts | 2X-Small | 2-4 |
| 3-10 analysts | Small | 4-8 |
| 10-30 analysts | Medium | 8-16 |
| 30+ analysts | Large + scaling | 16-32+ |

---
*Notebook 33 of the Databricks Data Engineering series*
*Study Order: 33*

---
# Section 4: Lakeview Dashboards

## Building Dashboards in Databricks

Lakeview (formerly SQL Dashboards) provides a drag-and-drop dashboard builder.

### Dashboard Components

| Component | Description | Example |
|-----------|-------------|---------|
| **Counter** | Single metric | Total Revenue: $1.2M |
| **Bar chart** | Compare categories | Revenue by region |
| **Line chart** | Trends over time | Daily orders (30 days) |
| **Pie chart** | Proportions | Revenue by payment method |
| **Table** | Detailed data | Top 20 customers |
| **Filter** | Interactive controls | Date range, region picker |

### Parameterized Queries

```sql
-- Parameters become filter widgets in the dashboard
SELECT
    order_date,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue
FROM techmart_dw.orders
WHERE order_date BETWEEN :start_date AND :end_date
  AND status IN (:status_filter)
  AND customer_segment = :segment
GROUP BY order_date
ORDER BY order_date
```

### Auto-Refresh

```
Dashboard settings -> Schedule -> Every 1 hour
```

The dashboard automatically re-runs all queries on schedule.
Analysts always see fresh data without manual refresh.

### Sharing Dashboards

- **View-only link**: Share with stakeholders (no Databricks account needed)
- **Embed**: Embed in Confluence, Notion, or internal portals
- **Schedule email**: Send PDF snapshot daily/weekly
- **Alerts**: Notify when a metric crosses a threshold

In [0]:
%sql
-- Build a comprehensive Gold table for dashboards
-- This pre-aggregates data so dashboards load instantly

DROP TABLE IF EXISTS techmart_dw.gold_dashboard_metrics;

CREATE TABLE techmart_dw.gold_dashboard_metrics AS
SELECT
    o.order_date,
    YEAR(o.order_date) AS order_year,
    MONTH(o.order_date) AS order_month,
    c.customer_segment,
    p.category AS product_category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    ROUND(SUM(oi.line_total), 2) AS total_revenue,
    ROUND(AVG(oi.line_total), 2) AS avg_line_value,
    ROUND(SUM(oi.line_total) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
JOIN techmart_dw.order_items oi ON o.order_id = oi.order_id
JOIN techmart_dw.products p ON oi.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY o.order_date, YEAR(o.order_date), MONTH(o.order_date),
         c.customer_segment, p.category

In [0]:
%sql
-- Dashboard query 1: Revenue trend (last 30 days)
SELECT order_date, SUM(total_revenue) AS revenue, SUM(total_orders) AS orders
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= DATE_ADD(CURRENT_DATE(), -30)
GROUP BY order_date
ORDER BY order_date

In [0]:
%sql
-- Dashboard query 2: Revenue by segment and category
SELECT
    customer_segment,
    product_category,
    SUM(total_revenue) AS revenue,
    SUM(total_orders) AS orders
FROM techmart_dw.gold_dashboard_metrics
GROUP BY customer_segment, product_category
ORDER BY revenue DESC
LIMIT 20

In [0]:
%sql
-- Dashboard query 3: KPI summary (counter widgets)
SELECT
    SUM(total_revenue) AS total_revenue,
    SUM(total_orders) AS total_orders,
    COUNT(DISTINCT unique_customers) AS unique_customers,
    ROUND(SUM(total_revenue) / SUM(total_orders), 2) AS avg_order_value
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= DATE_ADD(CURRENT_DATE(), -30)

---
# Section 5: SQL Warehouse Optimization for Dashboards

## Key Optimizations

1. **Pre-aggregate in Gold layer** -- avoid full scans on every load
2. **Liquid Clustering** on filter columns (date, region, segment)
3. **Query result cache** -- automatic in SQL Warehouses (same query = instant)
4. **Auto-suspend** -- set to 10 minutes to avoid idle cost
5. **Serverless warehouse** -- instant start, no cluster management

## Warehouse Sizing Guide

| Team Size | Warehouse Size | Concurrent Queries |
|-----------|---------------|-------------------|
| 1-3 analysts | 2X-Small | 2-4 |
| 3-10 analysts | Small | 4-8 |
| 10-30 analysts | Medium | 8-16 |
| 30+ analysts | Large + scaling | 16-32+ |

---
*Notebook 33 of the Databricks Data Engineering series*
*Study Order: 33*

---
# Section 4: Lakeview Dashboards

## Building Dashboards in Databricks

Lakeview (formerly SQL Dashboards) provides a drag-and-drop dashboard builder.

### Dashboard Components

| Component | Description | Example |
|-----------|-------------|---------|
| **Counter** | Single metric | Total Revenue: $1.2M |
| **Bar chart** | Compare categories | Revenue by region |
| **Line chart** | Trends over time | Daily orders (30 days) |
| **Pie chart** | Proportions | Revenue by payment method |
| **Table** | Detailed data | Top 20 customers |
| **Filter** | Interactive controls | Date range, region picker |

### Parameterized Queries

```sql
-- Parameters become filter widgets in the dashboard
SELECT
    order_date,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue
FROM techmart_dw.orders
WHERE order_date BETWEEN :start_date AND :end_date
  AND status IN (:status_filter)
  AND customer_segment = :segment
GROUP BY order_date
ORDER BY order_date
```

### Auto-Refresh

```
Dashboard settings -> Schedule -> Every 1 hour
```

The dashboard automatically re-runs all queries on schedule.
Analysts always see fresh data without manual refresh.

### Sharing Dashboards

- **View-only link**: Share with stakeholders (no Databricks account needed)
- **Embed**: Embed in Confluence, Notion, or internal portals
- **Schedule email**: Send PDF snapshot daily/weekly
- **Alerts**: Notify when a metric crosses a threshold

In [0]:
%sql
-- Build a comprehensive Gold table for dashboards
-- This pre-aggregates data so dashboards load instantly

DROP TABLE IF EXISTS techmart_dw.gold_dashboard_metrics;

CREATE TABLE techmart_dw.gold_dashboard_metrics AS
SELECT
    o.order_date,
    YEAR(o.order_date) AS order_year,
    MONTH(o.order_date) AS order_month,
    c.customer_segment,
    p.category AS product_category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    ROUND(SUM(oi.line_total), 2) AS total_revenue,
    ROUND(AVG(oi.line_total), 2) AS avg_line_value,
    ROUND(SUM(oi.line_total) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
JOIN techmart_dw.order_items oi ON o.order_id = oi.order_id
JOIN techmart_dw.products p ON oi.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY o.order_date, YEAR(o.order_date), MONTH(o.order_date),
         c.customer_segment, p.category

In [0]:
%sql
-- Dashboard query 1: Revenue trend (last 30 days)
SELECT order_date, SUM(total_revenue) AS revenue, SUM(total_orders) AS orders
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= DATE_ADD(CURRENT_DATE(), -30)
GROUP BY order_date
ORDER BY order_date

In [0]:
%sql
-- Dashboard query 2: Revenue by segment and category
SELECT
    customer_segment,
    product_category,
    SUM(total_revenue) AS revenue,
    SUM(total_orders) AS orders
FROM techmart_dw.gold_dashboard_metrics
GROUP BY customer_segment, product_category
ORDER BY revenue DESC
LIMIT 20

In [0]:
%sql
-- Dashboard query 3: KPI summary (counter widgets)
SELECT
    SUM(total_revenue) AS total_revenue,
    SUM(total_orders) AS total_orders,
    COUNT(DISTINCT unique_customers) AS unique_customers,
    ROUND(SUM(total_revenue) / SUM(total_orders), 2) AS avg_order_value
FROM techmart_dw.gold_dashboard_metrics
WHERE order_date >= DATE_ADD(CURRENT_DATE(), -30)

---
# Section 5: SQL Warehouse Optimization for Dashboards

## Key Optimizations

1. **Pre-aggregate in Gold layer** -- avoid full scans on every load
2. **Liquid Clustering** on filter columns (date, region, segment)
3. **Query result cache** -- automatic in SQL Warehouses (same query = instant)
4. **Auto-suspend** -- set to 10 minutes to avoid idle cost
5. **Serverless warehouse** -- instant start, no cluster management

## Warehouse Sizing Guide

| Team Size | Warehouse Size | Concurrent Queries |
|-----------|---------------|-------------------|
| 1-3 analysts | 2X-Small | 2-4 |
| 3-10 analysts | Small | 4-8 |
| 10-30 analysts | Medium | 8-16 |
| 30+ analysts | Large + scaling | 16-32+ |

---
*Notebook 33 of the Databricks Data Engineering series*
*Study Order: 33*

---
*Notebook 38 of the Databricks Data Engineering series*
*Study Order: 33*